In [ ]:
!pip -q install rasterio

import os, time
import numpy as np
import rasterio
import PIL.Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from torchvision import models
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
from google.colab import drive
drive.mount('/content/drive')

!cp -r "/content/drive/MyDrive/Satellite_ML_Research_Project/Dataset/allBands" "/content"

dataset_path = "/content/allBands"
print("Dataset path exists:", os.path.exists(dataset_path))

class EuroSATLandOnly(Dataset):
    def __init__(self, root_dir, img_size=224):
        self.root_dir = root_dir
        self.img_size = img_size

        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        self.samples = []
        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            for f in os.listdir(cls_path):
                if f.lower().endswith(".tif"):
                    self.samples.append((os.path.join(cls_path, f), self.class_to_idx[cls]))

        self.transform = Compose([
            Resize((img_size, img_size)),
            ToTensor(),
            Normalize(mean=[0.485, 0.456, 0.406],
                      std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        with rasterio.open(path) as src:
            img = src.read().astype(np.float32)

        img = np.clip(img, 0, 10000) / 10000.0

        B02 = img[1]
        B03 = img[2]
        B04 = img[3]

        rgb = np.stack([B04, B03, B02], axis=-1)
        rgb_pil = PIL.Image.fromarray((rgb * 255).astype(np.uint8))

        x = self.transform(rgb_pil)
        y = torch.tensor(label, dtype=torch.long)
        return x, y


def get_loaders(dataset, batch_size=32):
    n = len(dataset)
    n_train = int(0.7 * n)
    n_val   = int(0.15 * n)
    n_test  = n - n_train - n_val

    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],
                                             generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader


dataset = EuroSATLandOnly(dataset_path, img_size=224)
print("Total images:", len(dataset))
print("Classes:", dataset.classes)

train_loader, val_loader, test_loader = get_loaders(dataset, batch_size=32)

num_classes = len(dataset.classes)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

best_val = 0.0
best_path = "land_resnet18_best.pth"
epochs = 8

for ep in range(1, epochs+1):
    t0 = time.time()

    model.train()
    correct, total, running_loss = 0, 0, 0.0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()

        running_loss += loss.item() * x.size(0)
        pred = out.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    train_acc = correct / total
    train_loss = running_loss / total

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    val_acc = correct / total
    dt = time.time() - t0

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), best_path)

    print(f"Epoch {ep}/{epochs} | Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Time: {dt:.1f}s")

print("Best Val Acc:", best_val*100)
print("Saved:", best_path)
torch.save(model.state_dict(), "land_resnet18_best.pth")
!cp land_resnet18_best.pth "/content/drive/MyDrive/Satellite_ML_Research_Project/Models/"
print("Saved to Drive ✅")


Device: cuda
Mounted at /content/drive
Dataset path exists: True
Total images: 27611
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 158MB/s]


Epoch 1/8 | Loss: 0.2324 | Train Acc: 92.59% | Val Acc: 84.59% | Time: 129.7s
Epoch 2/8 | Loss: 0.0931 | Train Acc: 96.88% | Val Acc: 95.60% | Time: 118.7s
Epoch 3/8 | Loss: 0.0548 | Train Acc: 98.27% | Val Acc: 96.50% | Time: 107.7s
Epoch 4/8 | Loss: 0.0473 | Train Acc: 98.44% | Val Acc: 94.93% | Time: 109.9s
Epoch 5/8 | Loss: 0.0349 | Train Acc: 98.83% | Val Acc: 89.91% | Time: 109.5s
Epoch 6/8 | Loss: 0.0324 | Train Acc: 99.04% | Val Acc: 94.76% | Time: 107.0s
Epoch 7/8 | Loss: 0.0277 | Train Acc: 99.09% | Val Acc: 97.15% | Time: 105.9s
Epoch 8/8 | Loss: 0.0234 | Train Acc: 99.32% | Val Acc: 97.15% | Time: 106.4s
Best Val Acc: 97.15044675199226
Saved: land_resnet18_best.pth
Saved to Drive ✅
